# 05 - building the main dataset

this is where we combine the trip data and weather into one clean dataset ready for modelling.

the main thing we had to be careful about: if you build the zone grid from the zones that appear in trip records, you silently drop the 105 zones that have no taxi activity. that messes up the lag features later. so we get the zone list from the shapefile instead, which gives us all 263.

In [ ]:
import duckdb
import pandas as pd
import geopandas as gpd
from pathlib import Path

ROOT         = Path('/Users/Xavier/Desktop/university/YEAR4/datascienceNO3')
DATA_DIR     = ROOT / 'data'
WEATHER_PATH = ROOT / 'weather' / 'weather_by_zone_2023_2024_clean.parquet'
SHP_PATH     = Path('/tmp/taxi_zones/taxi_zones/taxi_zones.shp')
OUT_PATH     = ROOT / 'exports' / 'demand_weather_joined_2023_2024.parquet'

CAB_FILES = {
    'yellow': DATA_DIR / 'clean_yellow.parquet',
    'green':  DATA_DIR / 'clean_green.parquet',
    'fhvhv':  DATA_DIR / 'clean_fhvhv.parquet',
}

WEATHER_FEATURES = [
    'temperature_2m', 'precipitation', 'snowfall',
    'windspeed_10m', 'weathercode',
]

load the zone list from the shapefile so we get all 263 zones, not just the ones that appear in trips.

In [ ]:
def load_all_zone_ids():
    print('loading zone list from shapefile...', flush=True)
    if not SHP_PATH.exists():
        import io, zipfile, urllib.request
        url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip'
        print('  shapefile not cached - downloading...', flush=True)
        with urllib.request.urlopen(url, timeout=60) as r:
            data = r.read()
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            z.extractall('/tmp/taxi_zones')

    gdf      = gpd.read_file(str(SHP_PATH))
    zone_ids = sorted(gdf['LocationID'].astype(int).tolist())
    print(f'  {len(zone_ids)} zones from shapefile')
    return zone_ids


all_zone_ids = load_all_zone_ids()

aggregate trip counts to hourly by zone, combining all three cab types.

In [ ]:
def aggregate_trips():
    print('aggregating trips...', flush=True)
    parts = []
    for cab, path in CAB_FILES.items():
        if not path.exists():
            print(f'  warning: {path.name} not found - skipping')
            continue
        print(f'  {cab}...', end=' ', flush=True)
        sql = f"""
            SELECT
                date_trunc('hour', pickup_datetime)::TIMESTAMP AS hour_slot,
                CAST(pu_zone_id AS INTEGER)                    AS zone_id,
                COUNT(*)                                       AS trip_count
            FROM read_parquet('{path}')
            WHERE pickup_datetime >= '2023-01-01'
              AND pickup_datetime <  '2025-01-01'
              AND pu_zone_id IS NOT NULL
            GROUP BY 1, 2
        """
        df = duckdb.execute(sql).fetchdf()
        print(f'{len(df):,} rows')
        parts.append(df)

    combined = (
        pd.concat(parts, ignore_index=True)
          .groupby(['hour_slot', 'zone_id'], as_index=False)['trip_count']
          .sum()
          .rename(columns={'trip_count': 'total_trips'})
    )
    print(f'  combined: {len(combined):,} rows, {combined["zone_id"].nunique()} unique zones')
    return combined


trips = aggregate_trips()

build the full 263-zone x hourly grid and fill any missing zone/hour combinations with 0 trips.

In [ ]:
def zero_fill(trips, all_zone_ids):
    print('zero-filling full hour x zone grid...', flush=True)
    all_hours     = pd.date_range('2023-01-01', '2024-12-31 23:00', freq='h')
    expected_rows = len(all_hours) * len(all_zone_ids)
    print(f'  grid size: {len(all_hours):,} hours x {len(all_zone_ids)} zones = {expected_rows:,} rows')

    idx = pd.MultiIndex.from_product(
        [all_hours, all_zone_ids],
        names=['hour_slot', 'zone_id']
    )
    full = (
        trips.set_index(['hour_slot', 'zone_id'])
             .reindex(idx, fill_value=0)
             .reset_index()
    )
    zero_rows = (full['total_trips'] == 0).sum()
    pct_zero  = zero_rows / len(full) * 100
    print(f'  zero-demand rows: {zero_rows:,} ({pct_zero:.1f}%)')
    return full


full_grid = zero_fill(trips, all_zone_ids)

join the weather data onto the demand grid. we left join so all demand rows are kept, and forward-fill any gaps in weather.

In [ ]:
def join_weather(df):
    print('loading and joining weather...', flush=True)
    weather = pd.read_parquet(WEATHER_PATH)
    weather['time'] = pd.to_datetime(weather['time'])
    weather = weather.rename(columns={'time': 'hour_slot', 'LocationID': 'zone_id'})
    weather = weather[['hour_slot', 'zone_id'] + WEATHER_FEATURES]
    print(f'  weather: {len(weather):,} rows, {weather["zone_id"].nunique()} zones')

    out = df.merge(weather, on=['hour_slot', 'zone_id'], how='left')

    missing = out[WEATHER_FEATURES].isnull().any(axis=1).sum()
    if missing:
        print(f'  {missing:,} rows missing weather - forward-filling per zone')
        for col in WEATHER_FEATURES:
            out[col] = out.groupby('zone_id')[col].transform(lambda s: s.ffill().bfill())
        still_missing = out[WEATHER_FEATURES].isnull().any(axis=1).sum()
        if still_missing:
            print(f'  {still_missing:,} rows still missing - filling with global median/mode')
            for col in WEATHER_FEATURES[:-1]:
                out[col].fillna(out[col].median(), inplace=True)
            out['weathercode'].fillna(out['weathercode'].mode()[0], inplace=True)
    else:
        print('  all rows have weather assigned')

    print(f'  joined: {len(out):,} rows, {out["zone_id"].nunique()} zones')
    return out


joined = join_weather(full_grid)

print(f'\nfinal checks:')
print(f'  zones:       {joined["zone_id"].nunique()} (expected 263)')
print(f'  rows:        {len(joined):,}')
nan_count = joined[WEATHER_FEATURES].isnull().sum().sum()
print(f'  nan weather: {nan_count}')
zero_zones = (joined.groupby('zone_id')['total_trips'].sum() == 0).sum()
print(f'  zero demand zones: {zero_zones}')

OUT_PATH.parent.mkdir(exist_ok=True)
joined.to_parquet(OUT_PATH, index=False)
print(f'\nsaved -> {OUT_PATH}')

## quick nan cleanup

this was used on an earlier combined parquet to strip any remaining nans from numeric columns before modelling. keeping it here for reference.

In [ ]:
import numpy as np
import pandas as pd

parquet_file = r'combined/post_clean_concat_2min_bucket.parquet'

df = pd.read_parquet(parquet_file)

df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')

# keep only 2022-2026
df = df[(df['pickup_datetime'] >= '2022-01-01') &
        (df['pickup_datetime'] <= '2026-12-31')]

# get rid of nans in numeric cols only
numeric_cols = df.select_dtypes(include='number').columns
df = df.dropna(subset=numeric_cols)

df.to_parquet(r'/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/combined/no_nans_concat_2min_bucket.parquet', index=False)